# NPE tuning stage 7

In [1]:
import numpy as np
from scipy import stats
import torch
from tqdm.auto import tqdm
import itertools
import torch
import torch.nn as nn
from torch.distributions import Uniform
import sbi
from sbi.utils.user_input_checks import MultipleIndependent
from sbi.neural_nets import posterior_nn
from sbi.neural_nets.embedding_nets import FCEmbedding
from sbi.inference import NPE_C
from sbi.diagnostics import run_sbc, check_sbc
import warnings
import sys
sys.path.append('../../pysimARG')
from discrete_uniform import DiscreteUniform
from LeaveLengthOut_NN import LeaveLengthOut_NN

torch_device = "cpu"

warnings.filterwarnings("ignore", category=UserWarning)

c:\Users\u2008181\likelihood-free\sbi_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load simulation data

Load genome data and clonal tree.

In [2]:
drop_col = range(16, 32)
theta_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta_sbc.csv', delimiter=",")
x_test = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x_sbc.csv', delimiter=",")
x_test = np.delete(x_test, drop_col, axis=1)
print(theta_test.shape, x_test.shape)

nan_row_test = np.where(np.isnan(x_test) | np.isinf(x_test))[0]
print(nan_row_test)

theta_test = np.delete(theta_test, nan_row_test, axis=0)
theta_test = torch.tensor(theta_test, device=torch_device)
theta_test = theta_test.to(torch.float32)
theta_test_numpy = theta_test.cpu().numpy()

x_test = np.delete(x_test, nan_row_test, axis=0)
x_test = torch.tensor(x_test, device=torch_device)
x_test = x_test.to(torch.float32)
x_test_numpy = x_test.cpu().numpy()

print(theta_test.shape, x_test.shape)

(1000, 3) (1000, 30)
[865 899]
torch.Size([998, 3]) torch.Size([998, 30])


In [3]:
theta1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta1.csv', delimiter=",")
x1 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x1.csv', delimiter=",")
theta2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/theta2.csv', delimiter=",")
x2 = np.loadtxt('../../data/ClonalOrigin/rho_and_theta/x2.csv', delimiter=",")

x = np.vstack([x1, x2])
x = np.delete(x, drop_col, axis=1)
theta = np.vstack([theta1, theta2])
print(theta.shape, x.shape)

nan_row = np.where(np.isnan(x) | np.isinf(x))[0]
print(nan_row)

theta = np.delete(theta, nan_row, axis=0)
theta = torch.tensor(theta[:10000, :], device=torch_device)
theta = theta.to(torch.float32)
theta_numpy = theta.cpu().numpy()

x = np.delete(x, nan_row, axis=0)
x = torch.tensor(x[:10000, :], device=torch_device)
x = x.to(torch.float32)
x_numpy = x.cpu().numpy()

print(theta.shape, x.shape)

(20000, 3) (20000, 30)
[  114   681   706  1448  2554  2818  7211  7282  7329  7392  8938  9827
  9973 10223 10788 12192 13567 14388 14653]
torch.Size([10000, 3]) torch.Size([10000, 30])


## Test functions

In [4]:
def SBC_KStest(ranks, num_posterior_samples, parameter_labels):
    num_dimensions = ranks.shape[1] 

    ks_results = []
    p_values = []
    for dim in range(num_dimensions):
        normalized_ranks = ranks[:, dim] / num_posterior_samples
        ks_stat, p_value = stats.kstest(normalized_ranks, 'uniform')
        ks_results.append(ks_stat)
        p_values.append(p_value)
    
    return ks_results, p_values

In [5]:
def mahalanobis_error(theta_est_post, theta_test_numpy):
    maha_errors = np.full((theta_est_post.shape[0]), np.nan)
    for i in range(theta_est_post.shape[0]):
        samples = theta_est_post[i]
        truth = theta_test_numpy[i]
        post_mean = np.mean(samples, axis=0)
        cov_matrix = np.cov(samples, rowvar=False)

        try:
            inv_cov = np.linalg.inv(cov_matrix)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular covariance matrix at index {i}, returning NaN.")
            continue
        
        diff = post_mean - truth
        maha_dist_sq = np.dot(np.dot(diff, inv_cov), diff)
        maha_errors[i] = np.sqrt(maha_dist_sq)
    return maha_errors

## Tuning setting

In [6]:
seeds = [1, 2, 3, 4, 5]
num_posterior_samples = 1000
stage6_num_outputs = 2
stage6_num_transforms = 8
stage6_array = np.array([[1, 128, 30, 5],
                         [4, 256, 30, 20],
                         [4,  48, 50, 10]])
print(stage6_array)

[[  1 128  30   5]
 [  4 256  30  20]
 [  4  48  50  10]]


In [7]:
stage6_indices = [0, 1, 2]
learning_rates = [0.0001, 0.0005, 0.001]
stop_epochs = [10, 20, 25]
training_batch_sizes = [100, 200, 300]
clip_max_norms = [None, 1, 5, 8]
stage7_array = np.array(list(itertools.product(stage6_indices,
                                               learning_rates,
                                               stop_epochs,
                                               training_batch_sizes,
                                               clip_max_norms)))
print(stage7_array)

[[0 0.0001 10 100 None]
 [0 0.0001 10 100 1]
 [0 0.0001 10 100 5]
 ...
 [2 0.001 25 300 1]
 [2 0.001 25 300 5]
 [2 0.001 25 300 8]]


In [8]:
stage7_p_values = np.full((stage7_array.shape[0], 3), np.nan)
stage7_D_stats = np.full((stage7_array.shape[0], 3), np.nan)
stage7_maha_errors = np.full((stage7_array.shape[0]), np.nan)
stage7_nll = np.full((stage7_array.shape[0]), np.nan)

In [9]:
split_indices = 54 * np.array([0, 1, 2, 3, 4, 5, 6])
print(split_indices)

[  0  54 108 162 216 270 324]


## Baseline NPE

In [10]:
prior_rho = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_theta = Uniform(low=torch.tensor([0.0]), high=torch.tensor([0.1]))
prior_L = DiscreteUniform(low=torch.tensor([100.0]), high=torch.tensor([10000.0]))
prior = MultipleIndependent(
    dists=[prior_rho, prior_theta, prior_L],
    validate_args=False,
    device=torch_device
)

In [11]:
# for k in range(stage7_array.shape[0]):
#     # Stage 6 configuration
#     stage6_config = stage6_array[stage7_array[k, 0]]
#     num_hidden_layers = stage6_config[0]
#     num_hiddens = stage6_config[1]
#     num_features = stage6_config[2]
#     num_bins = stage6_config[3]


#     # Stage 7 configuration
#     learning_rate = stage7_array[k, 1]
#     stop_epoch = stage7_array[k, 2]
#     training_batch_size = stage7_array[k, 3]
#     clip_max_norm = stage7_array[k, 4]
    
#     embedding_net = LeaveLengthOut_NN(
#         input_dim=30,
#         num_hiddens=num_hiddens,
#         num_hidden_layers=num_hidden_layers,
#         num_outputs=stage6_num_outputs)
#     neural_posterior = posterior_nn(
#         model="nsf",
#         embedding_net=embedding_net,
#         hidden_features=num_features,
#         num_transforms=stage6_num_transforms,
#         num_bins=num_bins
#     )
#     print(f"Running iteration {k}")
#     print(f"Stage 5 output dimension {stage6_num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}.")
#     print(f"Stage 6 transforms {stage6_num_transforms}, features {num_features}, bins {num_bins}.")
#     print(f"Stage 7 learning rate {learning_rate}, stop epoch {stop_epoch},")
#     print(f"training batch size {training_batch_size}, clip max norm {clip_max_norm}.")
#     print("-" * 50)
    
#     seed = seeds[0]
#     torch.manual_seed(seed)
#     np.random.seed(seed)

#     inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
#     density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
#         max_num_epochs=500,
#         training_batch_size=training_batch_size,
#         learning_rate=learning_rate,
#         stop_after_epochs=stop_epoch,
#         clip_max_norm=clip_max_norm
#     )
#     posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

#     theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
#     for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
#         theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
#                                             show_progress_bars=False, reject_outside_prior=False)
#         theta_est_post[j, :, :] = theta_post.detach().numpy()

#     parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
#     theta_test_expanded = theta_test.unsqueeze(1)
#     theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
#     theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
#     is_less_than_truth = theta_est_post_tensor < theta_test_expanded
#     ranks = torch.sum(is_less_than_truth, dim=1)

#     ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
#     stage7_p_values[k, :] = p_values
#     stage7_D_stats[k, :] = ks_results

#     stage7_maha_errors[k] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

#     lp = density_estimator_baseline.log_prob(theta_test, x_test)
#     stage7_nll[k] = -lp.detach().cpu().mean().item()

In [12]:
# np.save('../../data/NPE_tuning/stage7_p_values.npy', stage7_p_values)
# np.save('../../data/NPE_tuning/stage7_D_stats.npy', stage7_D_stats)
# np.save('../../data/NPE_tuning/stage7_maha_errors.npy', stage7_maha_errors)
# np.save('../../data/NPE_tuning/stage7_nll.npy', stage7_nll)

## Load results and find the top three

In [13]:
# for i in range(6):
#     print(f"Load and combine subset {i+1} of 6.")
#     start_index = split_indices[i]
#     end_index = split_indices[i + 1]
#     subset_p_values = np.load(f'../../data/NPE_tuning/stage7_p_values_{i+1}.npy')
#     subset_D_stats = np.load(f'../../data/NPE_tuning/stage7_D_stats_{i+1}.npy')
#     subset_maha_errors = np.load(f'../../data/NPE_tuning/stage7_maha_errors_{i+1}.npy')
#     subset_nll = np.load(f'../../data/NPE_tuning/stage7_nll_{i+1}.npy')

#     stage7_p_values[start_index:end_index, :] = subset_p_values[start_index:end_index, :]
#     stage7_D_stats[start_index:end_index, :] = subset_D_stats[start_index:end_index, :]
#     stage7_maha_errors[start_index:end_index] = subset_maha_errors[start_index:end_index]
#     stage7_nll[start_index:end_index] = subset_nll[start_index:end_index]

In [14]:
# np.save('../../data/NPE_tuning/stage7_p_values.npy', stage7_p_values)
# np.save('../../data/NPE_tuning/stage7_D_stats.npy', stage7_D_stats)
# np.save('../../data/NPE_tuning/stage7_maha_errors.npy', stage7_maha_errors)
# np.save('../../data/NPE_tuning/stage7_nll.npy', stage7_nll)

stage7_p_values = np.load('../../data/NPE_tuning/stage7_p_values.npy')
stage7_D_stats = np.load('../../data/NPE_tuning/stage7_D_stats.npy')
stage7_maha_errors = np.load('../../data/NPE_tuning/stage7_maha_errors.npy')
stage7_nll = np.load('../../data/NPE_tuning/stage7_nll.npy')

In [15]:
stage7_nll.shape

(324,)

In [16]:
indices = np.argpartition(stage7_nll, 8)[:8]
print(indices)

[ 24 232  28  12 244 248  32  16]


In [17]:
stage7_array[indices]

array([[0, 0.0001, 25, 100, None],
       [2, 0.0001, 20, 200, None],
       [0, 0.0001, 25, 200, None],
       [0, 0.0001, 20, 100, None],
       [2, 0.0001, 25, 200, None],
       [2, 0.0001, 25, 300, None],
       [0, 0.0001, 25, 300, None],
       [0, 0.0001, 20, 200, None]], dtype=object)

In [18]:
stage7_p_values_final = np.full((len(seeds), 3, 8), np.nan)
stage7_D_stats_final = np.full((len(seeds), 3, 8), np.nan)
stage7_maha_errors_final = np.full((len(seeds), 8), np.nan)
stage7_nll_final = np.full((len(seeds), 8), np.nan)

In [19]:
# for m in range(len(indices)):
#     k = indices[m]
#     # Stage 6 configuration
#     stage6_config = stage6_array[stage7_array[k, 0]]
#     num_hidden_layers = stage6_config[0]
#     num_hiddens = stage6_config[1]
#     num_features = stage6_config[2]
#     num_bins = stage6_config[3]

#     # Stage 7 configuration
#     learning_rate = stage7_array[k, 1]
#     stop_epoch = stage7_array[k, 2]
#     training_batch_size = stage7_array[k, 3]
#     clip_max_norm = stage7_array[k, 4]
    
#     embedding_net = LeaveLengthOut_NN(
#         input_dim=30,
#         num_hiddens=num_hiddens,
#         num_hidden_layers=num_hidden_layers,
#         num_outputs=stage6_num_outputs)
#     neural_posterior = posterior_nn(
#         model="nsf",
#         embedding_net=embedding_net,
#         hidden_features=num_features,
#         num_transforms=stage6_num_transforms,
#         num_bins=num_bins
#     )
#     print(f"Running iteration {k}")
#     print(f"Stage 5 output dimension {stage6_num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}.")
#     print(f"Stage 6 transforms {stage6_num_transforms}, features {num_features}, bins {num_bins}.")
#     print(f"Stage 7 learning rate {learning_rate}, stop epoch {stop_epoch},")
#     print(f"training batch size {training_batch_size}, clip max norm {clip_max_norm}.")
#     print("-" * 50)

#     for i in range(len(seeds)):
#         print(f"Running seed {seeds[i]}...")
#         seed = seeds[i]
#         torch.manual_seed(seed)
#         np.random.seed(seed)

#         inference_baseline = NPE_C(prior=prior, density_estimator=neural_posterior, device=torch_device)
#         density_estimator_baseline = inference_baseline.append_simulations(theta, x).train(
#             max_num_epochs=500,
#             training_batch_size=training_batch_size,
#             learning_rate=learning_rate,
#             stop_after_epochs=stop_epoch,
#             clip_max_norm=clip_max_norm
#         )
#         posterior_baseline = inference_baseline.build_posterior(density_estimator_baseline)

#         theta_est_post = np.full((theta_test.shape[0], num_posterior_samples, 3), np.nan)
#         for j in tqdm(range(theta_test.shape[0]), desc="Sampling posterior"):
#             theta_post = posterior_baseline.sample((num_posterior_samples,), x=x_test[j, :],
#                                                 show_progress_bars=False, reject_outside_prior=False)
#             theta_est_post[j, :, :] = theta_post.detach().numpy()

#         parameter_labels = [r"for $\rho_s$", r"for $\theta_s$", r"for L"]
#         theta_test_expanded = theta_test.unsqueeze(1)
#         theta_est_post_tensor = torch.tensor(theta_est_post, device=torch_device)
#         theta_est_post_tensor = theta_est_post_tensor.to(torch.float32)
#         is_less_than_truth = theta_est_post_tensor < theta_test_expanded
#         ranks = torch.sum(is_less_than_truth, dim=1)

#         ks_results, p_values = SBC_KStest(ranks, num_posterior_samples, parameter_labels)
#         stage7_p_values_final[i, :, m] = p_values
#         stage7_D_stats_final[i, :, m] = ks_results

#         stage7_maha_errors_final[i, m] = np.mean(mahalanobis_error(theta_est_post, theta_test_numpy))

#         lp = density_estimator_baseline.log_prob(theta_test, x_test)
#         stage7_nll_final[i, m] = -lp.detach().cpu().mean().item()

In [20]:
# for i in range(1, 5):
#     print(f"Load and combine subset {i} of 4.")
#     script_index = [2*(i-1), 2*(i-1) + 1]
#     subset_p_values = np.load(f'../../data/NPE_tuning/stage7_p_values_{i}.npy')
#     subset_D_stats = np.load(f'../../data/NPE_tuning/stage7_D_stats_{i}.npy')
#     subset_maha_errors = np.load(f'../../data/NPE_tuning/stage7_maha_errors_{i}.npy')
#     subset_nll = np.load(f'../../data/NPE_tuning/stage7_nll_{i}.npy')

#     stage7_p_values_final[:, :, script_index] = subset_p_values[:, :, script_index]
#     stage7_D_stats_final[:, :, script_index] = subset_D_stats[:, :, script_index]
#     stage7_maha_errors_final[:, script_index] = subset_maha_errors[:, script_index]
#     stage7_nll_final[:, script_index] = subset_nll[:, script_index]

In [21]:
# np.save('../../data/NPE_tuning/stage7_p_values_final.npy', stage7_p_values_final)
# np.save('../../data/NPE_tuning/stage7_D_stats_final.npy', stage7_D_stats_final)
# np.save('../../data/NPE_tuning/stage7_maha_errors_final.npy', stage7_maha_errors_final)
# np.save('../../data/NPE_tuning/stage7_nll_final.npy', stage7_nll_final)

stage7_p_values_final = np.load('../../data/NPE_tuning/stage7_p_values_final.npy')
stage7_D_stats_final = np.load('../../data/NPE_tuning/stage7_D_stats_final.npy')
stage7_maha_errors_final = np.load('../../data/NPE_tuning/stage7_maha_errors_final.npy')
stage7_nll_final = np.load('../../data/NPE_tuning/stage7_nll_final.npy')

In [22]:
print(np.mean(stage7_nll_final, axis=0))
print(np.median(stage7_nll_final, axis=0))

[-4.99587951 -4.65708852 -4.98224425 -5.6923933  -5.20268021 -4.18552217
 -5.04552832 -5.03965397]
[-5.40769148 -4.73018885 -5.41107273 -5.77185678 -5.05811024 -4.21452284
 -5.25641298 -5.30325127]


In [23]:
print(np.mean(stage7_maha_errors_final, axis=0))
print(np.median(stage7_maha_errors_final, axis=0))

[1.792651   1.1950933  1.50635623 1.40632507 2.17717172 1.34778729
 1.39571858 1.39590685]
[1.42683987 1.18273814 1.47369068 1.38452554 1.65027566 1.29515195
 1.38544663 1.40887475]


In [24]:
print(np.argsort(np.mean(stage7_nll_final, axis=0)))
print(np.argsort(np.median(stage7_nll_final, axis=0)))

[3 4 6 7 0 2 1 5]
[3 2 0 7 6 4 1 5]


In [25]:
print(np.argsort(np.mean(stage7_maha_errors_final, axis=0)))
print(np.argsort(np.median(stage7_maha_errors_final, axis=0)))

[1 5 6 7 3 2 0 4]
[1 5 3 6 7 0 2 4]


In [26]:
print(np.median(stage7_p_values_final, axis=0))

[[2.07881376e-01 3.42477024e-01 3.12707096e-01 1.45934358e-01
  2.72076279e-01 2.72485614e-01 2.85339952e-01 1.84722930e-01]
 [4.21278477e-02 1.11555308e-01 1.92885157e-02 1.16144948e-01
  8.18459168e-02 2.29273036e-01 1.25340268e-01 1.28813207e-01]
 [9.77763918e-36 0.00000000e+00 1.23042626e-31 8.79040760e-36
  0.00000000e+00 4.90454463e-44 4.07328174e-27 2.45660871e-34]]


In [30]:
print(stage7_p_values_final[:, :, 3])

[[2.71442831e-02 6.00116551e-01 2.72089021e-31]
 [1.45934358e-01 1.46908134e-01 0.00000000e+00]
 [9.11456868e-02 1.46548927e-03 0.00000000e+00]
 [2.85622627e-01 1.16144948e-01 8.79040760e-36]
 [6.46857381e-01 9.66738686e-02 2.19740517e-35]]


In [27]:
indices[3]

np.int64(12)

In [28]:
final_indices = [3]
print("Final Configurations:")
print("-" * 50)
for i in final_indices:
    k = indices[i]
    # Stage 6 configuration
    stage6_config = stage6_array[stage7_array[k, 0]]
    num_hidden_layers = stage6_config[0]
    num_hiddens = stage6_config[1]
    num_features = stage6_config[2]
    num_bins = stage6_config[3]

    # Stage 7 configuration
    learning_rate = stage7_array[k, 1]
    stop_epoch = stage7_array[k, 2]
    training_batch_size = stage7_array[k, 3]
    clip_max_norm = stage7_array[k, 4]
    print(f"Index {k}")
    print(f"Stage 5 output dimension {stage6_num_outputs}, hidden layers {num_hidden_layers}, hidden units {num_hiddens}.")
    print(f"Stage 6 transforms {stage6_num_transforms}, features {num_features}, bins {num_bins}.")
    print(f"Stage 7 learning rate {learning_rate}, stop epoch {stop_epoch},")
    print(f"training batch size {training_batch_size}, clip max norm {clip_max_norm}.")
    print("-" * 50)

Final Configurations:
--------------------------------------------------
Index 12
Stage 5 output dimension 2, hidden layers 1, hidden units 128.
Stage 6 transforms 8, features 30, bins 5.
Stage 7 learning rate 0.0001, stop epoch 20,
training batch size 100, clip max norm None.
--------------------------------------------------
